# Pre-deforestation


In [7]:
import warnings
warnings.filterwarnings("ignore")
%pip install bmipy

  Using cached bmipy-2.0.1-py3-none-any.whl.metadata (2.7 kB)
Using cached bmipy-2.0.1-py3-none-any.whl (8.4 kB)
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 8.3 MB/s  0:00:00

   -------- ------------------------------- 1/5 [pathspec]
   -------- ------------------------------- 1/5 [pathspec]
   ------------------------ --------------- 3/5 [black]
   ------------------------ --------------- 3/5 [black]
   ------------------------ --------------- 3/5 [black]
   ------------------------ --------------- 3/5 [black]
   ------------------------ --------------- 3/5 [black]
   ------------------------ --------------- 3/5 [black]
   -------------------------------- ------- 4/5 [bmipy]
   ---------------------------------------- 5/5 [bmipy]

Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
print(os.getcwd())

import hbv_bmi
print("module imported")

from hbv_bmi import HBV_Bmi
print("class imported")

c:\Github repositories\teaching-materials\book\exercise_journal_ann_laura
module imported
class imported


In [1]:
# ============================================================
# 0. Imports
# ============================================================
import numpy as np
import pandas as pd
import xarray as xr
import json
from pathlib import Path
import matplotlib.pyplot as plt

from hbv_bmi import HBV_Bmi


# ============================================================
# 1. Paths and periods
# ============================================================
data_dir = Path("./Data")

precip_nc = data_dir / "manning_ERA5_precip_daily.nc"
evap_nc   = data_dir / "manning_ERA5_evap_daily.nc"   # use only if this is POTENTIAL evaporation
discharge_file = data_dir / "5202080_Q_Day.Cmd.txt"

shape_area = 6642 * 1e6  # m²

exp_start = "2014-01-01"
exp_end   = "2025-01-01"

pre_start  = "2014-01-01"
pre_end    = "2019-10-01"

post_start = "2020-03-01"
post_end   = "2025-01-01"


# ============================================================
# 2. Load discharge
# ============================================================
obs = pd.read_csv(
    discharge_file,
    delimiter=";",
    skiprows=36,
    header=0,
    encoding="cp1252"
)

obs.columns = ["Date", "Time", "Q"]
obs["Q"] = pd.to_numeric(obs["Q"], errors="coerce")
obs["Date"] = pd.to_datetime(obs["Date"])
obs = obs.set_index("Date").drop(columns=["Time"]).sort_index()

# convert m3/s to mm/day
if obs["Q"].max() > 50:
    obs["Q"] = obs["Q"] * 86400 / shape_area * 1000.0

obs.loc[obs["Q"] < 0, "Q"] = np.nan
obs.loc[obs["Q"] > 4000, "Q"] = np.nan
obs = obs.loc[exp_start:exp_end]


# ============================================================
# 3. Read forcing time axis from netCDF
# ============================================================
with xr.open_dataset(precip_nc) as ds:
    forcing_time = pd.to_datetime(ds["time"].values)

forcing_time = forcing_time[(forcing_time >= pd.Timestamp(exp_start)) &
                            (forcing_time <= pd.Timestamp(exp_end))]

# align obs to forcing dates
obs = obs.reindex(forcing_time)


# ============================================================
# 4. Objective function
# ============================================================
def calc_metrics(sim_series, obs_series):
    df = pd.concat([sim_series.rename("sim"), obs_series.rename("obs")], axis=1).dropna()

    if len(df) < 10:
        return np.nan, np.nan, np.nan

    sim = df["sim"].values
    obs = df["obs"].values

    if np.allclose(np.var(obs), 0):
        return np.nan, np.nan, np.nan

    sim_clip = np.clip(sim, 1e-6, None)
    obs_clip = np.clip(obs, 1e-6, None)

    nse = 1 - np.sum((sim - obs) ** 2) / np.sum((obs - np.mean(obs)) ** 2)
    lognse = 1 - np.sum((np.log(sim_clip) - np.log(obs_clip)) ** 2) / np.sum((np.log(obs_clip) - np.mean(np.log(obs_clip))) ** 2)

    De = np.sqrt((1 - nse) ** 2 + (1 - lognse) ** 2)
    return nse, lognse, De


# ============================================================
# 5. HBV run function
# ============================================================
config_path = Path("./hbv_config.json")

base_config = {
    "precipitation_file": str(precip_nc),
    "potential_evaporation_file": str(evap_nc),
    "initial_storage": "0,100,0,5",
    "parameters": ""
}

def run_hbv(parameter_vector):
    base_config["parameters"] = ",".join(str(p) for p in parameter_vector)

    with open(config_path, "w") as f:
        json.dump(base_config, f)

    model = HBV_Bmi()
    model.initialize(str(config_path))

    n_steps = model.end_timestep
    Q_sim = np.zeros(n_steps, dtype=float)

    for t in range(n_steps):
        model.update()
        Q_sim[t] = model.Q

    model.finalize()

    sim = pd.Series(Q_sim, index=forcing_time[:n_steps], name="Qsim")
    sim = sim.loc[exp_start:exp_end]
    return sim


# ============================================================
# 6. Parameter ranges
# ============================================================
par_names = ["Imax", "Ce", "Sumax", "beta", "Pmax", "Tlag", "Kf", "Ks"]

p_min = np.array([0.0,   0.4,   40.0, 1.0,   0.001, 1.0,  0.01,   0.0001])
p_max = np.array([8.0,   0.8,  800.0, 2.5,   0.3,   3.0,  0.1,    0.01])

N = 200   # use more samples than 20

np.random.seed(42)
parameters = np.random.uniform(p_min[:, None], p_max[:, None], size=(len(par_names), N))


# ============================================================
# 7. Random search calibration on PRE period
# ============================================================
results = []

obs_pre = obs.loc[pre_start:pre_end, "Q"]
obs_post = obs.loc[post_start:post_end, "Q"]

for i in range(N):
    par = parameters[:, i]

    try:
        sim = run_hbv(par)

        sim_pre = sim.loc[pre_start:pre_end]
        sim_post = sim.loc[post_start:post_end]

        nse_pre, lognse_pre, De_pre = calc_metrics(sim_pre, obs_pre)
        nse_post, lognse_post, De_post = calc_metrics(sim_post, obs_post)

        results.append({
            "idx": i,
            "Imax": par[0],
            "Ce": par[1],
            "Sumax": par[2],
            "beta": par[3],
            "Pmax": par[4],
            "Tlag": par[5],
            "Kf": par[6],
            "Ks": par[7],
            "NSE_pre": nse_pre,
            "logNSE_pre": lognse_pre,
            "De_pre": De_pre,
            "NSE_post": nse_post,
            "logNSE_post": lognse_post,
            "De_post": De_post
        })

    except Exception as e:
        results.append({
            "idx": i,
            "Imax": par[0],
            "Ce": par[1],
            "Sumax": par[2],
            "beta": par[3],
            "Pmax": par[4],
            "Tlag": par[5],
            "Kf": par[6],
            "Ks": par[7],
            "NSE_pre": np.nan,
            "logNSE_pre": np.nan,
            "De_pre": np.nan,
            "NSE_post": np.nan,
            "logNSE_post": np.nan,
            "De_post": np.nan
        })

results_df = pd.DataFrame(results)

# remove failed runs
results_df = results_df.dropna(subset=["De_pre"]).sort_values("De_pre").reset_index(drop=True)


# ============================================================
# 8. Best parameter set
# ============================================================
best = results_df.iloc[0]

best_par = np.array([
    best["Imax"], best["Ce"], best["Sumax"], best["beta"],
    best["Pmax"], best["Tlag"], best["Kf"], best["Ks"]
])

print("Best parameter set (calibrated on pre-deforestation):")
for name, val in zip(par_names, best_par):
    print(f"{name:>5s} = {val:.5f}")

print("\nPerformance of best set:")
print(f"Pre  NSE    = {best['NSE_pre']:.3f}")
print(f"Pre  logNSE = {best['logNSE_pre']:.3f}")
print(f"Pre  De     = {best['De_pre']:.3f}")
print(f"Post NSE    = {best['NSE_post']:.3f}")
print(f"Post logNSE = {best['logNSE_post']:.3f}")
print(f"Post De     = {best['De_post']:.3f}")


# ============================================================
# 9. Run model with best parameters
# ============================================================
sim_best = run_hbv(best_par)

# combined dataframe
hydro = pd.concat([obs["Q"], sim_best.rename("Qsim")], axis=1)


# ============================================================
# 10. Plot full, pre, post
# ============================================================
def plot_period(start, end, title, sim_color="tab:blue"):
    df = hydro.loc[start:end]

    plt.figure(figsize=(12, 4))
    plt.plot(df.index, df["Q"], color="black", label="Observed", linewidth=1.2)
    plt.plot(df.index, df["Qsim"], color=sim_color, label="Simulated", linewidth=1.2)
    plt.title(title)
    plt.ylabel("Discharge [mm/day]")
    plt.xlabel("Date")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_period(exp_start, exp_end, "HBV simulation - full period", "tab:green")
plot_period(pre_start, pre_end, "HBV calibration period (pre-deforestation)", "tab:blue")
plot_period(post_start, post_end, "HBV validation period (post-deforestation)", "tab:red")


# ============================================================
# 11. Show top 10 runs
# ============================================================
cols_show = ["idx", "NSE_pre", "logNSE_pre", "De_pre", "NSE_post", "logNSE_post", "De_post"] + par_names
print("\nTop 10 parameter sets:")
print(results_df[cols_show].head(10))

ValueError: found the following matches with the input file in xarray's IO backends: ['netcdf4', 'h5netcdf']. But their dependencies may not be installed, see:
https://docs.xarray.dev/en/stable/user-guide/io.html 
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html

In [4]:
# ============================================================
# 0. Imports
# ============================================================
import numpy as np
import pandas as pd
import xarray as xr
import json
from pathlib import Path
import matplotlib.pyplot as plt

from hbv_bmi import HBV_Bmi


# ============================================================
# 1. Paths and periods
# ============================================================
data_dir = Path("./Data")

precip_nc = data_dir / "manning_ERA5_precip_daily.nc"
evap_nc   = data_dir / "manning_ERA5_evap_daily.nc"
discharge_file = data_dir / "5202080_Q_Day.Cmd.txt"

shape_area = 6642 * 1e6  # m²

exp_start = "2014-01-01"
exp_end   = "2025-01-01"

pre_start  = "2014-01-01"
pre_end    = "2019-10-01"

post_start = "2020-03-01"
post_end   = "2025-01-01"


# ============================================================
# 2. Load discharge and clean
# ============================================================
obs = pd.read_csv(
    discharge_file,
    delimiter=';',
    skiprows=36,
    header=0,
    encoding='cp1252'
)
obs.columns = ["Date", "Time", "Q"]
obs["Q"] = pd.to_numeric(obs["Q"], errors="coerce")
obs = obs.dropna(subset=["Q"])
obs["Date"] = pd.to_datetime(obs["Date"])
obs = obs.set_index("Date").drop(columns=["Time"]).sort_index()

# m³/s → mm/day if needed
if obs["Q"].max() > 50:
    obs["Q"] = obs["Q"] * 86400 / shape_area * 1000.0

obs.loc[obs["Q"] > 4000, "Q"] = np.nan
obs.loc[obs["Q"] < 0, "Q"] = np.nan
obs = obs.loc[exp_start:exp_end]


# ============================================================
# 3. Precompute time index from forcing (no per-step datetime)
# ============================================================
ds_precip = xr.open_dataset(precip_nc)
time_index = pd.to_datetime(ds_precip["time"].values)
ds_precip.close()

mask_exp = (time_index >= pd.to_datetime(exp_start)) & (time_index <= pd.to_datetime(exp_end))
time_index = time_index[mask_exp]

# Align obs to this time index
obs = obs.reindex(time_index)


# ============================================================
# 4. Objective function (NSE, logNSE, De)
# ============================================================
def calibration_objective(model_df, obs_df, start_cal, end_cal):
    hydro = pd.concat(
        [model_df.reindex(obs_df.index, method="ffill"), obs_df],
        axis=1
    )
    hydro = hydro.loc[start_cal:end_cal]

    sim = hydro["model"].values.copy()
    obs = hydro["Q"].values.copy()

    # Simple interpolation for NaNs in obs
    for i in range(1, len(obs) - 1):
        if np.isnan(obs[i]) and not np.isnan(obs[i - 1]) and not np.isnan(obs[i + 1]):
            obs[i] = 0.5 * (obs[i - 1] + obs[i + 1])

    mask = ~np.isnan(obs)
    sim = sim[mask]
    obs = obs[mask]

    if len(obs) < 3:
        return np.nan, np.nan, np.nan

    sim_clipped = np.clip(sim, 1e-6, None)
    obs_clipped = np.clip(obs, 1e-6, None)

    nse = 1.0 - np.sum((sim - obs) ** 2) / np.sum((obs - obs.mean()) ** 2)
    log_nse = 1.0 - np.sum((np.log(sim_clipped) - np.log(obs_clipped)) ** 2) / \
                    np.sum((np.log(obs_clipped) - np.log(obs_clipped).mean()) ** 2)
    De = np.sqrt((1.0 - nse) ** 2 + (1.0 - log_nse) ** 2)
    return nse, log_nse, De


# ============================================================
# 5. Ensemble setup
# ============================================================
p_min_initial = np.array([0., 0.4,  40., 1.0, 0.001, 1.,  0.01, 0.0001])
p_max_initial = np.array([8., 0.8, 800., 2.5, 0.3,   3.,  0.1,  0.01])

N = 20  # reduced for speed
parameters = np.zeros((8, N))

np.random.seed(6)
for i in range(8):
    parameters[i, :] = np.random.uniform(p_min_initial[i], p_max_initial[i], N)


# ============================================================
# 6. One reusable JSON config file
# ============================================================
config_path = Path("./hbv_config.json")

base_config = {
    "precipitation_file": str(precip_nc),
    "potential_evaporation_file": str(evap_nc),
    "initial_storage": "0,100,0,5",
    "parameters": ""  # overwritten each iteration
}

with open(config_path, "w") as f:
    json.dump(base_config, f)


# ============================================================
# 7. Run ensemble (HBV_Bmi, optimized usage)
# ============================================================
objectives = []

for n in range(N):
    par = parameters[:, n]

    # Update parameters and overwrite same small JSON file
    base_config["parameters"] = ",".join(str(p) for p in par)
    with open(config_path, "w") as f:
        json.dump(base_config, f)

    model = HBV_Bmi()
    model.initialize(str(config_path))

    n_steps = model.end_timestep
    Q_m = np.zeros(n_steps, dtype=float)

    # Fast loop: only update + read Q, no datetime/xarray/list appends
    for t in range(n_steps):
        model.update()
        Q_m[t] = model.Q

    model_time = pd.date_range(exp_start, periods=n_steps, freq="D")
    model_df = pd.DataFrame({"model": Q_m}, index=model_time)
    model_df = model_df.loc[exp_start:exp_end]

    nse, log_nse, De = calibration_objective(
        model_df,
        obs[["Q"]],
        pre_start,
        pre_end
    )
    objectives.append((nse, log_nse, De))


# ============================================================
# 8. Best parameters
# ============================================================
objectives = np.array(objectives)
De_vals = objectives[:, 2]
best_idx = np.nanargmin(De_vals)
best_par = parameters[:, best_idx]

print("Best ensemble member index:", best_idx)
print("Best parameters (Imax, Ce, Sumax, beta, Pmax, Tlag, Kf, Ks):")
print(best_par)

SUmax_pre = best_par[2]
print("SUmax pre-deforestation =", SUmax_pre, "mm")


# ============================================================
# 9. Run HBV_Bmi with best parameters over full period
# ============================================================
base_config["parameters"] = ",".join(str(p) for p in best_par)
with open(config_path, "w") as f:
    json.dump(base_config, f)

model = HBV_Bmi()
model.initialize(str(config_path))

n_steps = model.end_timestep
Q_best = np.zeros(n_steps, dtype=float)

for t in range(n_steps):
    model.update()
    Q_best[t] = model.Q

model_time = pd.date_range(exp_start, periods=n_steps, freq="D")
sim_full = pd.DataFrame({"model": Q_best}, index=model_time)
sim_full = sim_full.loc[exp_start:exp_end]

obs_df = obs[["Q"]]


# ============================================================
# 10. Metrics: pre and post
# ============================================================
def print_metrics(label, start, end):
    sim = sim_full.loc[start:end]["model"]
    ob = obs_df.loc[start:end]["Q"]
    nse, lognse, De = calibration_objective(
        sim.to_frame("model"),
        ob.to_frame("Q"),
        start,
        end
    )
    print(f"\n{label}")
    print("  NSE    =", nse)
    print("  logNSE =", lognse)
    print("  De     =", De)
    return nse, lognse, De

pre_nse, pre_lognse, pre_De = print_metrics("Pre-deforestation (2014–2019-10-01):", pre_start, pre_end)
post_nse, post_lognse, post_De = print_metrics("Post-deforestation (2020–2025-01-01):", post_start, post_end)


# ============================================================
# 11. Plots
# ============================================================
def plot_period(title, start, end, color):
    sim = sim_full.loc[start:end]["model"]
    ob = obs_df.loc[start:end]["Q"]
    plt.figure(figsize=(12,4))
    plt.plot(ob.index, ob.values, label="Obs", color="k")
    plt.plot(sim.index, sim.values, label="HBV_Bmi", color=color)
    plt.title(title)
    plt.ylabel("Q [mm/day]")
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_period("Pre-deforestation (2014–2019-10-01)", pre_start, pre_end, "tab:blue")
plot_period("Post-deforestation (2020–2025-01-01)", post_start, post_end, "tab:orange")


ModuleNotFoundError: No module named 'bmipy'

POST DEFORESTATION

In [ ]:

# Post-deforestation (2020-03-01 → 2025-01-01)
post2_start = "2020-03-01"
post2_end   = "2025-01-01"


post2_sim = sim_full.loc[post2_start:post2_end]
post2_obs = obs.loc[post2_start:post2_end][["Q"]]

post2_nse, post2_lognse, post2_De = calibration_objective(
    post2_sim, post2_obs, post2_start, post2_end
)

print("\nPost-deforestation (extended) metrics:")
print("  NSE    =", post2_nse)
print("  logNSE =", post2_lognse)
print("  De     =", post2_De)

# Plot
plt.figure(figsize=(12,4))
plt.plot(post2_obs.index, post2_obs["Q"], label="Obs (post-extended)", color="k")
plt.plot(post2_sim.index, post2_sim["model"], label="HBV (post-extended)", color="tab:red")
plt.title("Post-deforestation (2020–2025): modeled vs observed")
plt.ylabel("Q [mm/day]")
plt.legend()
plt.tight_layout()
plt.show()
